# Lyra — Build seed dataset

Estrazione brani dalle playlist pubbliche Spotify → `data/seed_tracks.json` (+ `.csv`).
Implementa `engine/prompt_build_dataset.md`. Step successivi (NON qui): chiamate Musixmatch, verifica preview audio.

**Setup credenziali (`.env`):** `SPOTIFY_CLIENT_ID`, `SPOTIFY_CLIENT_SECRET`, `SPOTIFY_REDIRECT_URI`
(app creata su [developer.spotify.com](https://developer.spotify.com/dashboard)).

**Login utente (una tantum):** leggere i brani di una playlist ora richiede l'**Authorization Code flow** —
Client Credentials dà 401 sull'endpoint `/items`. Al **primo run** si apre il browser per il login Spotify;
il token resta in cache (`.cache`, gitignorato), quindi il login lo fai una sola volta.
⚠️ Il `SPOTIFY_REDIRECT_URI` nel `.env` deve combaciare **esattamente** con un Redirect URI salvato nelle impostazioni dell'app.

**Ordine di esecuzione:** Input playlist → Definizioni → Run di test → Run completa → Ispezione.

## Input playlist

In [ ]:
# Playlist sorgente: { url_completo : provider }.
# NB: Spotify (app in Development mode) legge SOLO le playlist DI TUA PROPRIETA.
# Le playlist di altri utenti danno 403, le editoriali (37i9...) tornano vuote.
# -> Per usare una playlist altrui: duplicala nel tuo account (diventi owner) e
#    incolla qui il link della TUA copia, nella sezione "ESTERNE DUPLICATE".
playlist_links = {
    # --- Le tue 29 playlist ---
    "https://open.spotify.com/playlist/2dYQIt4XgN42p6QMydjQVl": "spotify",  # Ref per live
    "https://open.spotify.com/playlist/0KQM0MG46u99JZ4IZbAmSW": "spotify",  # X kurt
    "https://open.spotify.com/playlist/0BgS6TJ45xqZm4lmmIsIEw": "spotify",  # sessioni di ascolto
    "https://open.spotify.com/playlist/3c2ZZP7Ur2quGfoWYlESzt": "spotify",  # train that ass
    "https://open.spotify.com/playlist/1CTshqBfyrg70kAbYw3QOO": "spotify",  # axel, e solitudine
    "https://open.spotify.com/playlist/6lYMhRsCMV5K2o2LQtSuNy": "spotify",  # hug me
    "https://open.spotify.com/playlist/5tsZxffqwU8dk0wyGdfMWO": "spotify",  # crushin' <3
    "https://open.spotify.com/playlist/5IKDb9OcUbqswAoKLon5sC": "spotify",  # da choreo
    "https://open.spotify.com/playlist/6rvQ4toVUm5r2kpw4mHDfo": "spotify",  # ivory coast
    "https://open.spotify.com/playlist/4z4HQOunsBapFgAh3r7efa": "spotify",  # berlin00
    "https://open.spotify.com/playlist/4l8QoEsWa09NmMrD5nqVSI": "spotify",  # afro jazz//lo-fi
    "https://open.spotify.com/playlist/5cldsXvMk6OTJ1yc7Z7STW": "spotify",  # baddie me
    "https://open.spotify.com/playlist/55V5YoczIF8RSB4QJx4Ndu": "spotify",  # body language // seduzione
    "https://open.spotify.com/playlist/2gUZACCbi4zGsA8b9aoKeY": "spotify",  # ref
    "https://open.spotify.com/playlist/3oCrFdAFRjslLkJ0m9SQnK": "spotify",  # reference afro rock
    "https://open.spotify.com/playlist/4X3F4ihIgeBIUmEoR6VHcu": "spotify",  # Drifting (Esp mix)
    "https://open.spotify.com/playlist/6AmHsXj9f7gdJpJBspvKVS": "spotify",  # intimate
    "https://open.spotify.com/playlist/2tXLJTXts54khJUqOAWEzf": "spotify",  # Xfactor
    "https://open.spotify.com/playlist/2RE8YBSh4JxdCoOzwhQo5D": "spotify",  # mood
    "https://open.spotify.com/playlist/1hUx9kGUuoPv7Z2fKC7tFL": "spotify",  # loading
    "https://open.spotify.com/playlist/6ZzGusvy302Ih1g0rgUi3t": "spotify",  # references for new era
    "https://open.spotify.com/playlist/3PKs7CrGjro3FxaBhcxtAR": "spotify",  # Vibez hip hip new gen
    "https://open.spotify.com/playlist/1RIOdjJcFbSRwsVxO9sosh": "spotify",  # Jam
    "https://open.spotify.com/playlist/5c7a2JuMNh49cttHBsQ3Lt": "spotify",  # BO0h
    "https://open.spotify.com/playlist/1aIubcnp6Be7o4LhTW7n3P": "spotify",  # Boh
    "https://open.spotify.com/playlist/5StWfcT6PLwkvJmK7f27Nr": "spotify",  # #MyMood
    "https://open.spotify.com/playlist/0Qn8xYX4NAKn3VoO14vgrn": "spotify",  # Fantastico!
    "https://open.spotify.com/playlist/6Cs9asm6Vj7NJNwesLo2g5": "spotify",  # Smooth!
    "https://open.spotify.com/playlist/2SRaoHV5gubuW0xDhp5v5b": "spotify",  # Study!

    # --- ESTERNE DUPLICATE (aggiungi qui i link delle TUE copie) ---
    # "https://open.spotify.com/playlist/XXXXXXXXXXXXXXXXXXXXXX": "spotify",  # ex-Karim 🇫🇷
}

In [ ]:
# =============================================================================
# Lyra — dataset builder (seed tracks from Spotify playlists)
# Implements engine/prompt_build_dataset.md.
#
# This cell only DEFINES helpers + the orchestrator. Run them in the cells below.
# It does NOT call Musixmatch, does NOT download audio/previews, and does NOT
# persist any Musixmatch content — only Spotify metadata for our own seed list.
#
# AUTH NOTE: Spotify requires USER auth (Authorization Code flow) to read
# /playlists/{id}/items. First run opens a browser to log in once; the token is
# cached in .cache (gitignored). With a Development-mode app you can read only
# YOUR OWN playlists — other users' playlists and /tracks return 403, editorial
# (37i9...) playlists come back empty.
# =============================================================================
from __future__ import annotations

import os
import csv
import json
import time
import logging
from pathlib import Path
from urllib.parse import urlparse

import spotipy
from spotipy.oauth2 import SpotifyOAuth
from spotipy.exceptions import SpotifyException
from dotenv import load_dotenv

# ---- logging (step 5: INFO progress, WARNING skips, ERROR failures) ---------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("lyra.dataset")

SAVE_EVERY = 50      # step 6: flush partial results to disk every N tracks
MAX_RETRIES = 3      # step 5: retries on Spotify 429 rate-limit


# ---- Spotify client ---------------------------------------------------------
def get_spotify_client() -> spotipy.Spotify:
    """Authorization Code Flow (user login) — required by Spotify to read
    /playlists/{id}/items. First run opens a browser to log in; the token is
    cached in .cache so you only log in once. Needs SPOTIFY_CLIENT_ID /
    SPOTIFY_CLIENT_SECRET / SPOTIFY_REDIRECT_URI (redirect must match the app
    settings exactly)."""
    load_dotenv()
    client_id = os.environ.get("SPOTIFY_CLIENT_ID")
    client_secret = os.environ.get("SPOTIFY_CLIENT_SECRET")
    redirect_uri = os.environ.get("SPOTIFY_REDIRECT_URI", "http://127.0.0.1:3000/callback")
    if not client_id or not client_secret:
        raise RuntimeError(
            "Missing SPOTIFY_CLIENT_ID / SPOTIFY_CLIENT_SECRET. "
            "Copy .env.example to .env and fill in your credentials."
        )
    auth = SpotifyOAuth(
        client_id=client_id,
        client_secret=client_secret,
        redirect_uri=redirect_uri,
        scope="playlist-read-private playlist-read-collaborative",
        cache_path=".cache",
        open_browser=True,
    )
    # retries=0: we handle 429 ourselves with explicit backoff (_with_retry)
    return spotipy.Spotify(auth_manager=auth, retries=0)


def _with_retry(fn, *args, **kwargs):
    """Call a spotipy method, retrying on 429 with exponential backoff."""
    for attempt in range(MAX_RETRIES):
        try:
            return fn(*args, **kwargs)
        except SpotifyException as exc:
            if exc.http_status == 429 and attempt < MAX_RETRIES - 1:
                headers = getattr(exc, "headers", None) or {}
                wait = int(headers.get("Retry-After", 2 ** (attempt + 1)))
                log.warning("Rate limited (429). Retry %d/%d in %ds.",
                            attempt + 1, MAX_RETRIES - 1, wait)
                time.sleep(wait)
                continue
            raise


# ---- step 2: parsing --------------------------------------------------------
def parse_spotify_playlist_id(url: str) -> str:
    """Extract the playlist id from a Spotify URL, dropping any ?query tracking
    (si / pi / nd / dlsi ...). Also accepts a spotify:playlist:<id> URI.
    e.g. .../playlist/37i9dQZF1DXcBWIGoYBM5M?si=... -> 37i9dQZF1DXcBWIGoYBM5M"""
    if url.startswith("spotify:playlist:"):
        return url.split(":")[-1]
    path = urlparse(url).path  # urlparse strips the ?si=...&pi=... query string
    marker = "/playlist/"
    if marker not in path:
        raise ValueError(f"Not a Spotify playlist URL: {url}")
    return path.split(marker, 1)[1].split("/")[0]


# ---- step 3: extraction -----------------------------------------------------
def extract_tracks_from_playlist(sp_client, playlist_id, source_name) -> list[dict]:
    """Return a list of track dicts from one playlist. Handles pagination (pages
    of 100), skips local files (id is None), tolerates missing ISRC."""
    tracks: list[dict] = []
    results = _with_retry(sp_client.playlist_items, playlist_id, limit=100,
                          additional_types=("track",))
    while results:
        for item in results.get("items", []):
            # Spotify moved the track payload from item["track"] to item["item"]
            # (2025/26 schema change); keep item["track"] as a fallback. NOTE: do
            # NOT pass a market here — with a market the track comes back null.
            t = item.get("item") or item.get("track")
            if not isinstance(t, dict):
                continue  # removed / unavailable entry (or a bare boolean flag)
            if item.get("is_local") or t.get("is_local") or t.get("id") is None:
                continue  # local file added to the playlist — skip (step 5)

            isrc = (t.get("external_ids") or {}).get("isrc")
            title = t.get("name")
            artists = t.get("artists") or [{}]
            artist = artists[0].get("name")  # first artist only, by design

            # skip corrupt records: no ISRC AND no usable (artist, title)
            if not isrc and not (artist and title):
                continue

            album = t.get("album") or {}
            tracks.append({
                "isrc": isrc,
                "spotify_id": t.get("id"),
                "artist": artist,
                "title": title,
                "album": album.get("name"),
                "duration_ms": t.get("duration_ms"),
                "release_date": album.get("release_date"),
                # popularity is NOT in the playlist-items payload anymore; filled
                # later by enrich_with_popularity() (best-effort, may stay None).
                "spotify_popularity": t.get("popularity"),
                "playlist_count": 1,
                "playlist_sources": [source_name],
            })

        # pagination: fetch the next page if present
        results = _with_retry(sp_client.next, results) if results.get("next") else None

    log.info("  extracted %d tracks from %s (%s)", len(tracks), source_name, playlist_id)
    return tracks


# ---- popularity enrichment (best-effort) ------------------------------------
def enrich_with_popularity(sp_client, tracks) -> list[dict]:
    """Fill spotify_popularity via batched /tracks calls (50 ids each). The
    playlist-items endpoint no longer returns popularity. Degrades silently if
    /tracks is forbidden for this app (leaves popularity as None)."""
    ids = [t["spotify_id"] for t in tracks if t.get("spotify_id")]
    pop: dict = {}
    for i in range(0, len(ids), 50):
        try:
            for tr in (_with_retry(sp_client.tracks, ids[i:i + 50]).get("tracks") or []):
                if tr:
                    pop[tr["id"]] = tr.get("popularity")
        except SpotifyException as exc:
            log.warning("Popularity enrichment unavailable (%s) — leaving it None.",
                        exc.http_status)
            return tracks
    for t in tracks:
        if t["spotify_id"] in pop:
            t["spotify_popularity"] = pop[t["spotify_id"]]
    return tracks


# ---- step 4: deduplication --------------------------------------------------
def dedup_tracks(tracks) -> list[dict]:
    """Deduplicate by ISRC (primary key); fallback to (artist.lower, title.lower)
    when ISRC is missing. On a duplicate, DON'T overwrite — bump playlist_count
    and extend playlist_sources with the new playlist."""
    merged: dict = {}
    for t in tracks:
        if t["isrc"]:
            key = ("isrc", t["isrc"])
        else:
            key = ("at", (t["artist"] or "").lower().strip(),
                         (t["title"] or "").lower().strip())

        if key in merged:
            existing = merged[key]
            for src in t["playlist_sources"]:
                if src not in existing["playlist_sources"]:
                    existing["playlist_sources"].append(src)
            existing["playlist_count"] = len(existing["playlist_sources"])
        else:
            merged[key] = dict(t)  # copy so we don't mutate the input
    return list(merged.values())


# ---- step 6: saving ---------------------------------------------------------
def save_outputs(tracks, output_dir):
    """Write data/seed_tracks.json and an equivalent .csv for quick inspection."""
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    json_path = out / "seed_tracks.json"
    csv_path = out / "seed_tracks.csv"

    json_path.write_text(
        json.dumps(tracks, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    fields = ["isrc", "spotify_id", "artist", "title", "album", "duration_ms",
              "release_date", "spotify_popularity", "playlist_count",
              "playlist_sources"]
    with csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for t in tracks:
            row = dict(t)
            row["playlist_sources"] = ";".join(t["playlist_sources"])  # flatten list
            writer.writerow(row)
    return json_path, csv_path


# ---- orchestrator -----------------------------------------------------------
def build_dataset(playlists_dict, output_dir="data") -> list[dict]:
    """Extract → enrich → dedup → save seed tracks from a {playlist_url: provider}
    dict. Returns the deduplicated list of track dicts and writes JSON + CSV."""
    sp = get_spotify_client()

    # stable symbolic source name per link (matches the spec's output example)
    sources = {url: f"link_playlist_{i}" for i, url in enumerate(playlists_dict)}

    all_tracks: list[dict] = []
    processed = skipped = 0
    n = len(playlists_dict)

    for i, (url, provider) in enumerate(playlists_dict.items()):
        source_name = sources[url]

        # Apple Music: not supported in this version (paid Apple Developer acct)
        if provider != "spotify":
            log.warning("Skipping %s link %s: not supported in this version. "
                        "Please convert to Spotify.", provider, url)
            skipped += 1
            continue

        try:
            pid = parse_spotify_playlist_id(url)
        except ValueError as exc:
            log.warning("Skipping unparseable link %s: %s", url, exc)
            skipped += 1
            continue

        try:
            log.info("[%d/%d] %s -> %s", i + 1, n, source_name, pid)
            tracks = extract_tracks_from_playlist(sp, pid, source_name)
        except SpotifyException as exc:
            if exc.http_status in (403, 404):
                log.warning("Playlist not accessible (%s) — likely owned by "
                            "another user or editorial: %s", exc.http_status, url)
            else:
                log.error("Failed on %s: %s", url, exc)
            skipped += 1
            continue

        prev_total = len(all_tracks)
        all_tracks.extend(tracks)
        processed += 1

        # step 6: checkpoint whenever we cross a multiple of SAVE_EVERY tracks
        if len(all_tracks) // SAVE_EVERY > prev_total // SAVE_EVERY:
            partial = dedup_tracks(all_tracks)
            save_outputs(partial, output_dir)
            log.info("  ...checkpoint saved (%d unique so far)", len(partial))

    # best-effort popularity, then final dedup + save
    all_tracks = enrich_with_popularity(sp, all_tracks)
    total_before = len(all_tracks)
    unique = dedup_tracks(all_tracks)
    save_outputs(unique, output_dir)

    # step 7: final report
    no_isrc = sum(1 for t in unique if not t["isrc"])
    c1 = sum(1 for t in unique if t["playlist_count"] == 1)
    c2 = sum(1 for t in unique if t["playlist_count"] == 2)
    c3 = sum(1 for t in unique if t["playlist_count"] >= 3)
    print("\n" + "=" * 48)
    print(f"Playlists processed: {processed}/{n} ({skipped} skipped)")
    print(f"Total tracks before dedup: {total_before}")
    print(f"Unique tracks (deduplicated): {len(unique)}")
    print(f"Tracks without ISRC: {no_isrc}")
    print("Distribution of playlist_count:")
    print(f"  1 playlist:  {c1} tracks")
    print(f"  2 playlists: {c2} tracks")
    print(f"  3+ playlists: {c3} tracks")
    print("=" * 48)
    return unique

In [ ]:
# --- Run di test su 2-3 link prima del run completo (vedi nota finale dello spec) ---
test_links = dict(list(playlist_links.items())[:3])
test_seed = build_dataset(test_links, output_dir="data_test")
test_seed[:3]

In [ ]:
# --- Run completa: tutte le playlist di input -> data/seed_tracks.{json,csv} ---
seed = build_dataset(playlist_links, output_dir="data")
print(f"\nTotal unique seed tracks: {len(seed)}")
seed[:3]

In [ ]:
# --- Ispezione del dataset generato (data/seed_tracks.json) ---
import json
import pandas as pd
from pathlib import Path

df = pd.read_json(Path("data") / "seed_tracks.json")
print(f"Totale brani: {len(df)}")
print(f"Senza ISRC:   {df['isrc'].isna().sum()}")
print(f"Artisti unici: {df['artist'].nunique()}")

# distribuzione playlist_count (in quante playlist compare ogni brano)
print("\nDistribuzione playlist_count:")
print(df["playlist_count"].value_counts().sort_index().to_string())

# popularity Spotify (0-100)
print("\nPopularity — describe:")
print(df["spotify_popularity"].describe().round(1).to_string())

# i 10 brani più "trasversali" (in più playlist) e i top per popularity
print("\nTop 10 per playlist_count:")
display(df.sort_values("playlist_count", ascending=False)
          .head(10)[["artist", "title", "playlist_count", "spotify_popularity"]])

print("Top 10 per popularity:")
display(df.sort_values("spotify_popularity", ascending=False)
          .head(10)[["artist", "title", "spotify_popularity", "playlist_count"]])

df.head()